# NB_hospital_daily_summary — Gold Layer

Builds `adb_caresync.gold.hospital_daily_summary`: one row per hospital per calendar day. Aggregates lab test counts and rates per hospital-day, enriches with hospital name and calendar attributes, and computes a day-over-day test volume change using a LAG window. Full rebuild on every run.

In [0]:
%sql
-- Ensure the gold schema exists before writing
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold

In [0]:
# Verify all required source silver tables are accessible
source_tables = [
    "adb_caresync.silver.lab_results",
    "adb_caresync.silver.hospitals",
    "adb_caresync.silver.calendar",
]

for t in source_tables:
    count = spark.table(t).count()
    schema = spark.table(t).dtypes
    print(f"✅ {t}: {count:,} rows | columns: {[c[0] for c in schema]}")

In [0]:
%sql
-- ─────────────────────────────────────────────────────────────
-- Target  : adb_caresync.gold.hospital_daily_summary
-- Grain   : One row per hospital per calendar day
-- Sources : silver.lab_results (driving, grouped by hospital_id + test_date)
--           silver.hospitals   (LEFT JOIN on hospital_id)
--           silver.calendar    (LEFT JOIN on date_key)
-- Windows : LAG(total_tests) PARTITION BY hospital_id ORDER BY summary_date
-- Load    : FULL — CREATE OR REPLACE on every run
-- ─────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary AS

WITH base AS (
    -- Step 1: aggregate lab_results per hospital per day; enrich via joins
    SELECT
        lr.hospital_id,
        h.hospital_name,
        lr.test_date                                                          AS summary_date,
        cal.day_name,
        cal.is_weekend,
        COUNT(*)                                                              AS total_tests,
        COUNT(DISTINCT lr.patient_id)                                         AS unique_patients_tested,
        COUNT(DISTINCT lr.doctor_id)                                          AS active_doctor_count,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)            AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)            AS critical_test_count
    FROM      adb_caresync.silver.lab_results  lr
    LEFT JOIN adb_caresync.silver.hospitals    h
           ON lr.hospital_id = h.hospital_id
    LEFT JOIN adb_caresync.silver.calendar     cal
           ON CAST(DATE_FORMAT(lr.test_date, 'yyyyMMdd') AS INT) = cal.date_key
    GROUP BY
        lr.hospital_id, h.hospital_name,
        lr.test_date,
        cal.day_name, cal.is_weekend
),

with_rates AS (
    -- Step 2: derived rate columns + LAG for previous-day total
    SELECT
        *,
        CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2))         AS abnormal_rate_pct,
        CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2))         AS critical_rate_pct,
        LAG(total_tests) OVER (
            PARTITION BY hospital_id
            ORDER BY     summary_date
        )                                                                      AS prev_day_total_tests
    FROM base
)

-- Step 3: day-over-day % change (NULL when no prior day or prior day had 0 tests)
SELECT
    hospital_id,
    hospital_name,
    summary_date,
    day_name,
    is_weekend,
    total_tests,
    unique_patients_tested,
    active_doctor_count,
    abnormal_test_count,
    critical_test_count,
    abnormal_rate_pct,
    critical_rate_pct,
    prev_day_total_tests,
    CASE
        WHEN prev_day_total_tests IS NULL OR prev_day_total_tests = 0 THEN NULL
        ELSE CAST((total_tests - prev_day_total_tests) / prev_day_total_tests * 100 AS DECIMAL(6,2))
    END AS test_volume_change_pct
FROM with_rates

In [0]:
record_count = spark.table("adb_caresync.gold.hospital_daily_summary").count()
dbutils.notebook.exit(str(record_count))